In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "LTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,108.87,108.89,108.81,108.89,110.284,2025-09-01 00:00:59.999999+00:00,12004.09154,159,78.063,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,108.90,108.98,108.90,108.98,143.091,2025-09-01 00:01:59.999999+00:00,15584.65490,141,131.541,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.002019,0.001122,0.000897,NaN,NaN
2,2025-09-01 00:02:00+00:00,108.98,108.98,108.86,108.91,60.387,2025-09-01 00:02:59.999999+00:00,6577.87439,143,7.238,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000402,0.000827,-0.000425,NaN,NaN
3,2025-09-01 00:03:00+00:00,108.91,108.93,108.87,108.88,356.131,2025-09-01 00:03:59.999999+00:00,38780.88348,170,263.167,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.001426,0.000064,-0.001490,NaN,NaN
4,2025-09-01 00:04:00+00:00,108.88,108.88,108.66,108.67,382.463,2025-09-01 00:04:59.999999+00:00,41584.76818,246,118.148,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.010746,-0.003152,-0.007594,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 06:08:43,946] A new study created in memory with name: no-name-eb2894f7-4319-4fe6-b1c7-49a9df7a4dcb


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:04<?, ?it/s]

Best trial: 0. Best value: 0.0223029:   0%|          | 0/50 [00:04<?, ?it/s]

Best trial: 0. Best value: 0.0223029:   2%|▏         | 1/50 [00:04<04:04,  5.00s/it]

...

Best trial: 32. Best value: 0.0298343:  98%|█████████▊| 49/50 [02:53<00:06,  6.18s/it]

Best trial: 32. Best value: 0.0298343:  98%|█████████▊| 49/50 [02:53<00:06,  6.18s/it]

Best trial: 32. Best value: 0.0298343: 100%|██████████| 50/50 [02:53<00:00,  4.79s/it]

Best trial: 32. Best value: 0.0298343: 100%|██████████| 50/50 [02:53<00:00,  3.47s/it]

[I 2026-03-20 06:11:37,527] Trial 49 finished with value: 0.01651180975952671 and parameters: {'n_estimators': 200, 'max_depth': 11, 'learning_rate': 0.01587877966335136, 'subsample': 0.6785083952154263, 'colsample_bytree': 0.9977300707169632, 'min_child_weight': 5, 'reg_alpha': 4.998411508306453e-08, 'reg_lambda': 0.001608210711648249}. Best is trial 32 with value: 0.029834292370303713.

[optuna] best trial
value: 0.029834
params:
  n_estimators: 400
  max_depth: 11
  learning_rate: 0.0015891997776799004
  subsample: 0.7262207011589449
  colsample_bytree: 0.674657154997091
  min_child_weight: 3
  reg_alpha: 3.7585775978173047e-07
  reg_lambda: 0.004117224251641722


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 4.70s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...



===== RESULTS =====
Train IC:      0.725068
Test IC:       -0.007514
Train Rank IC: 0.177542
Test Rank IC:  0.030929
Train RMSE:    0.002789
Test RMSE:     0.001990


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
trend_x_imb         0.094859
trend_strength      0.080458
vol_30              0.053751
imbalance_5         0.047919
vol_15              0.043974
num_trades_mom_5    0.042093
imbalance_15        0.040400
volume_mom_5        0.037956
vol_ratio_5_30      0.035132
dom_sin             0.034159
dist_ma_15_z        0.029016
month_sin           0.028800
imbalance           0.027370
mom_10              0.024609
hour_sin            0.024391
mr_x_vol            0.022922
mom_x_imb           0.021272
range_ratio         0.020946
mom_5               0.020856
volume_z            0.020717
range_15            0.019519
dist_ma_30          0.019260
mom_60              0.016172
atr_norm            0.014952
trades_z            0.014876
month_cos           0.014167
vol_regime_ratio    0.013686
hour_cos            0.012059
dow_sin             0.010904
vol_5               0.010873
mom_3               0.010508
dom_cos             0.010449
mom_30              0.009898
macd_hist  

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/LTCUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/LTCUSDT__h5_model.joblib
[saved] features -> models/xgb/LTCUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/LTCUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/LTCUSDT__h5_meta.json
